In [62]:
import numpy as np
import string

In [63]:
np.random.seed(42)

In [64]:
initial = {}  # start of a phrase
first_order = {} # second word only
second_order= {}

In [65]:
def remove_punctuation(s):
  return s.translate(str.maketrans('','', string.punctuation))

In [66]:
def add2dict(d, k, v):
  if k not in d:
    d[k] = []
  d[k].append(v)

In [68]:
for line in open('/content/robert_frost.txt'):
  tokens = remove_punctuation(line.rstrip().lower()).split()

  T = len(tokens)
  for i in range(T):
    t = tokens[i]
    if i == 0:

      initial[t] = initial.get(t, 0.) + 1
    else:
      t_1 = tokens[i-1]
      if i == T-1:
        add2dict(second_order, (t_1,t), 'END')
      if i == 1:
        add2dict(first_order, t_1, t)
      else:
        t_2 = tokens[i-2]
        add2dict(second_order, (t_2, t_1), t)



In [69]:
# normalize the distributions
initial_total = sum(initial.values())
for t, c in initial.items():
  initial[t] = c/initial_total


In [70]:
def list2predict(ts):

  d = {}
  n = len(ts)
  for t in ts:
    d[t] = d.get(t, 0.) + 1
  for t, c in d.items():
    d[t] = c /n
  return d


In [71]:
for t_1, ts in first_order.items():
  first_order[t_1] = list2predict(ts)


In [72]:
for k, ts in second_order.items():
  second_order[k] = list2predict(ts)

In [84]:
import numpy as np

def sample_word(d):
    p0 = np.random.random()
    cumulative = 0

    for t, p in d.items():
        cumulative += p
        if p0 < cumulative:
            return t

    # If no word is returned within the loop (should not happen if probabilities sum to 1)
    assert False, "Error: No word selected"

def generate():
    for i in range(4):
        sentence = []
        w0 = sample_word(initial)
        sentence.append(w0)

        w1 = sample_word(first_order[w0])
        sentence.append(w1)

        while True:
            try:
                w2 = sample_word(second_order.get((w0, w1), {'END': 1.0}))
                if w2 == 'END':
                    break
                sentence.append(w2)
                w0 = w1
                w1 = w2
            except AssertionError:
                break

        print(' '.join(sentence))

# # Example usage:
# initial = {'apple': 0.4, 'orange': 0.6}
# first_order = {'apple': {'banana': 0.2, 'cherry': 0.8}, 'orange': {'kiwi': 0.5, 'pear': 0.5}}
# second_order = {('apple', 'banana'): {'grape': 1.0}, ('apple', 'cherry'): {'END': 1.0}, ('orange', 'kiwi'): {'lemon': 0.3, 'peach': 0.7}}

generate()


apple cherry
orange kiwi
orange pear
orange kiwi
